# USB-6289 driver — how to use it

A tutorial for the QCoDeS driver in `NIUSB_6289_v2`. It goes from the simplest
thing (set a voltage, read it back) to the hardest (record for hours against an
atomic clock and watch it live), and every function is explained where it is
first used.

**Hardware assumed** — a loopback and the rubidium 1 pps:

| signal | terminal | ground |
|---|---|---|
| AO 0 → AI 0 | 15 → 1 | AO GND 16 → AI GND 3 |
| 1 pps → PFI 9 | 83 | D GND 82 |

**Restart the kernel** whenever you edit the driver. Python caches imported
modules, so an edited file has no effect until you do — this has caused more
confusion than any actual bug.

## 0. Imports

The driver is split into files by job:

| module | what it holds | needs hardware? |
|---|---|---|
| `usb6289` | `USB6289`, the instrument you construct | yes |
| `postprocess` | reading and analysing recorded runs | **no** — numpy only |
| `livescope` | viewing a run while it is being written | **no** — numpy only |

`postprocess` and `livescope` import nothing but numpy, so you can analyse a
week-long recording on a laptop that has never had NI-DAQmx installed.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from NIUSB_6289_v2.usb6289 import USB6289
from NIUSB_6289_v2.postprocess import (load_long_run, load_segment, load_run,
                                       iter_segments, channels_of, edge_report,
                                       clean_edges, times_from_edges,
                                       load_markers, marker_times,
                                       run_header, trigger_time,
                                       to_hdf5)
from NIUSB_6289_v2.livescope import LiveScope, RunFiles

## 1. Connect

```python
USB6289(name, device="Dev1")
```

* `name` — the QCoDeS instrument name. Must be unique in the session; making a
  second `USB6289("daq")` raises. If you re-run this cell, call `daq.close()`
  first.
* `device` — the DAQmx device name, as shown in NI MAX.

On construction the driver **asks the board what it is** — how many channels,
which counters, the maximum rates, the available input ranges — and builds its
validators from the answer. It does not trust a data sheet.

In [ ]:
daq = USB6289("daq", device="Dev1")

### The instrument tree

Everything hangs off `daq` in three subsystems:

```
daq                     the board
├── daq.ai              analog INPUT task    (one clock, one trigger)
│   ├── daq.ai.ai0 … ai31        32 input pins
│   └── daq.ai.channels          the same 32, as a group
├── daq.pps             the 1 pps counter    (no channels: one counter, one job)
├── daq.marker          the event counter    (the same mechanism, other wire)
└── daq.ao              analog OUTPUT task   (one clock, one waveform)
    ├── daq.ao.ao0 … ao3         4 output pins
    └── daq.ao.channels          the same 4, as a group
```

`pps` and `marker` are **the same mechanism** pointed at different wires: a
counter that records the scan index of every pulse on a PFI line. Point one at
the rubidium and you get atomic seconds — the *ruler*. Point the other at the
OPX and you get *events*. The board has exactly two counters and both are
spoken for; a start trigger needs none, because it is not timestamped — it
**defines** scan 0.

The rule for *where a setting lives*: *who does the hardware let decide it?*

* decided **per pin** → on the channel: `ai0.v_max`, `ao0.dc`
* decided **once per task** → on the subsystem: `daq.ai.rate`, `daq.ao.freq`
  (there is one ADC, so you cannot give two inputs different sample rates)
* needs **two subsystems** → on the instrument: `daq.clock_check`, `daq.long_run`

In [ ]:
daq.ai.channels.enabled()          # broadcast: read a parameter on all 32 at once

---
# 1. Loopback test

AO 0 wired to AI 0: the board generates a signal and measures its own output.
Everything downstream depends on this working.

## 2.1 Constant voltage

### `daq.ai.setup(...)` — configure the input in one call

```python
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=0.05)
```

* **first argument** says which pins are on **and** their range. It accepts
  * `"ai0"` — enable it, leave the range alone
  * `["ai0", "ai3"]` — several
  * `{"ai0": 2.0}` — a number means **± that**
  * `{"ai0": (-0.5, 1.5)}` — a pair means exactly that
  * each input has its own programmable gain, so mixing ranges is free
* `rate` — samples per second **per channel**
* `duration` — seconds per `acquire()`
* `trigger` — a terminal to wait for; `None` starts immediately
* **enabling is exclusive**: pins you do not name are switched off
* **returns the channel objects**, so `ai0, =` gives you a handle in one line

### `daq.ao.setup(...)` — configure the output

```python
ao0, = daq.ao.setup(channels_role={"ao0": {"dc": 0.0}})
```

Each pin has a **role**, because the board can drive it two different ways:

| role | meaning |
|---|---|
| `"off"` | not in the task |
| `"dc"` | held at a constant voltage by an untimed write |
| `"wave"` | generating the waveform — **at most one pin**, the board has one AO clock |

Going to `"off"` does *not* change the pin's voltage: an untimed output holds
its last value indefinitely. Set `dc(0.0)` first if you want it at zero.

In [ ]:
# configuration for AnalogInput
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=0.05)

# configuration for AnalogOutput: ao0 as a DC level, starting at 0 V
ao0, = daq.ao.setup(channels_role={"ao0": {"dc": 0.0}})   # name BOTH: setup
                                                    # is not exclusive

### `daq.check()` — validate before touching the hardware

```python
daq.check(verbose=True, strict=False)
```

Returns a list of `(level, where, message)`, empty if all is well.
`strict=True` raises on any error instead of returning.

It catches what no single validator can, because the constraint spans
parameters or lives in the driver:

* `amp + |offset|` against the ±10 V output range
* `rate × channels` against the board's **aggregate** limit
* a DIFF channel whose negative half is separately enabled
* a waveform buffer too big for the onboard FIFO
* a record too large to hold in RAM
* **a trigger terminal that cannot be routed** — it hands the task to DAQmx for
  verification, which programs the clocks without starting them

That last one is the prize: it is the failure that otherwise appears only when
the run starts, possibly hours later.

In [ ]:
daq.check()

### `daq.wiring()` — which screw terminals to connect

```python
rows = daq.wiring(verbose=True)
```

Prints a table **and returns it** as a list of dicts
(`role, channel, connection, terminal, signal, note`), so it goes straight into
a DataFrame or a JSON file recording what was physically plugged in.

It reads the **live** configuration, so it changes as you enable channels or
move the waveform. Every signal row is followed by its **return** row: on a
multiplexed board a floating input does not read zero, it reads whatever the
multiplexer left behind.

In [ ]:
daq.wiring()

### Where do I plug this in?

Two different questions, two tools.

**The static question** — "which terminal is `ai3`?" The answer never changes;
it is the connector.

| call | returns |
|---|---|
| `daq.pin_of("ai3")` | `10` — accepts `ai3`, `AI 3`, `Dev1/ai3`, `PFI8`, `AI GND` |
| `daq.signal_at(81)` | `'PFI 8/P2.0'` — the inverse: what *is* this terminal? |
| `daq.nearest_ground(15, "AO GND")` | `16` — the ground pin to actually wire to |
| `daq.pinout()` | the whole 128-pin connector, both halves |

`nearest_ground` takes the ground **family** — `"AI GND"`, `"AO GND"`,
`"D GND"` — and which one you use matters: returning a signal through the
wrong ground is how you get a mains hum you cannot explain.

**The live question** — "what do I plug in for what I have configured right
now?" That is `daq.wiring()`, further down. It changes as you enable channels
or move the waveform, and it pairs every signal with its return.

In [ ]:
print("ai3 is on terminal ", daq.pin_of("ai3"))
print("its AI GND is       ", daq.nearest_ground(daq.pin_of("ai3"), "AI GND"))
print("terminal 81 carries ", daq.signal_at(81))

daq.pinout()          # the full connector

### Which pins can be a trigger, and which can carry the 1 pps?

```python
daq.terminals(physical=True)     # only the ones a cable can reach
daq.terminals(physical=False)    # everything DAQmx will route
```

The board reports **two kinds** of routable terminal, and the difference
decides what each can be used for:

| kind | example | can an external cable reach it? |
|---|---|---|
| **physical** | `/Dev1/PFI8` → terminal 81 | **yes** |
| **internal** | `/Dev1/ai/SampleClock`, `/Dev1/Ctr0InternalOutput` | no — no pin exists |

* `ai.trigger` and `ao.trigger` accept **either**. An internal route is how one
  subsystem triggers another without a wire.
* `pps.terminal` must be **physical** — the rubidium's pulse arrives from
  outside the board, so it needs a pin.

Both parameters validate against this list, so a terminal that cannot be
routed is rejected at set time; and `daq.check()` goes further by handing the
task to DAQmx, which rejects routes that are legal in principle but conflict
with what else is configured.

### What if the trigger and the 1 pps are on *different* pins?

**Nothing breaks — that is the ordinary case.** They are independent settings:

* `ai.trigger` decides *when the acquisition starts*
* `pps.terminal` decides *where the atomic seconds arrive*

Putting both on PFI 8 is a convenience, not a requirement. It buys one thing:
the same edge starts the acquisition and marks an atomic second, so **scan 0
is an atomic second** and the first edge comes back at scan 0.

On separate pins, the first edge lands wherever it lands — somewhere in the
first second — and `times_from_edges` still puts `t = 0` at that first edge.
Samples taken *before* it come out **negative**, which is correct and
deliberate: they happened before the first atomic second you witnessed.

That is often what you want. With `ai.trigger` on the OPX marker and
`pps.terminal` on PFI 9, the first edge tells you exactly how far the next
atomic second was from the start of the shot — which is the number that puts a
QM experiment on the atomic timebase.

**One cost of sharing the pin**, worth knowing: the acquisition then waits for
the next 1 pps edge, so it starts up to 1 s late — and if the rubidium is off
or unplugged, it waits for the 60 s read timeout instead of starting at all.

In [ ]:
daq.terminals()                    # physical: a cable can reach these

# ai.trigger and ao.trigger can also use internal routes:
# daq.terminals(physical=False)

### So where can the 1 pps actually go?

**Any PFI line — PFI 0 to PFI 15.** All sixteen are on the digital connector
and any of them can clock a counter, so the choice is about wiring, not
capability.

`daq.terminals()` lists them with their **ground** pin, and the board is not
even-handed:

```
   /Dev1/PFI0    terminal  73   D GND  82 (9 pins away)
   /Dev1/PFI7    terminal  80   D GND  82 (2 pins away)
   /Dev1/PFI8    terminal  81   D GND  82 (adjacent)
   /Dev1/PFI9    terminal  83   D GND  82 (adjacent)
   /Dev1/PFI15   terminal  95   D GND  94 (adjacent)
```

**PFI 8–15 each sit next to a D GND pin. PFI 0–7 do not** — they all share the
single D GND at pin 82, up to nine screws away.

For an external TTL that matters twice over: a short return loop picks up less,
and two adjacent screws are simply easier to get right at the terminal block at
midnight. So for the rubidium, use **PFI 8–15**.

| line | signal pin | ground pin |
|---|---|---|
| PFI 8 | 81 | 82 |
| PFI 9 | 83 | 82 |
| PFI 10 | 85 | 84 |
| PFI 11 | 87 | 86 |
| PFI 12 | 89 | 88 |

The convention in this driver's docstrings is **PFI 8 for the trigger, PFI 9
for the 1 pps** — arbitrary, but consistent, and `daq.wiring()` prints whatever
you actually chose.

One thing that is *not* a constraint: the counter. `pps.counter` defaults to
`ctr1`, and any PFI line can clock either counter, so the two choices are
independent.

### Is any pin *dedicated* to the trigger, or to the 1 pps?

**No — and that is a property of this board, not an omission.** On an M Series
device every PFI line is fully routable to every timing signal: start trigger,
sample clock, counter source, counter gate. "Which pin is the trigger pin?" has
no answer beyond *the one you chose*.

(It was a real question on the older **E Series**, where PFI 0 *was* the AI
start trigger and nothing else. That is not this board — if you are reading an
older manual or an older script, that is the difference.)

What the manual *does* define is the pin each counter uses **when you do not
say otherwise**:

| line | default role | line | default role |
|---|---|---|---|
| PFI 3 | CTR 1 SRC | PFI 10 | CTR 0 AUX |
| PFI 4 | CTR 1 GATE | PFI 11 | CTR 1 AUX |
| PFI 8 | CTR 0 SRC | PFI 12 | CTR 0 OUT |
| PFI 9 | CTR 0 GATE | PFI 13 | CTR 1 OUT |
| | | PFI 14 | FREQ OUT |

These are **not reservations**. This driver routes the counter's sample clock
explicitly, so `pps.terminal` works on any PFI line including the ones above —
which is why PFI 9 is used here despite being CTR 0's default gate. They matter
only because a counter left on its defaults *in some other script* will quietly
take those lines.

`daq.terminal_roles()` puts it together: every PFI line, its pins, what **you**
are using it for right now, and what the board would use it for by default. It
reads the defaults back **from the device** where it can, falling back to the
table above.

In [ ]:
daq.terminal_roles()      # who is using what, and what the defaults are

### The sweep

* `ao0.dc(v)` — set the level. Applied **immediately**; no clock, nothing to
  start. Setting `dc` on a pin whose role is `"off"` promotes it to `"dc"`.
* `daq.ai.acquire()` — run **one** acquisition and return
  `{channel_name: array}`. This is the only call that touches the AI hardware.
* `time.sleep(0.1)` — let the output settle before measuring.

In [ ]:
v_min, v_max = -1.0, 1.0
step = 0.5

V = np.arange(v_min, v_max + step, step)
V

In [ ]:
for v in V:
    ao0.dc(v)                       # set the output
    time.sleep(0.1)                 # let it settle
    data = daq.ai.acquire()         # measure - returns {channel: array}
    trace = data["ai0"]

    print(f"set {v:+.2f} V  ->  read {trace.mean():+.4f} V "
          f"(+/- {trace.std():.4f}, {trace.size} samples)")

ao0.dc(0)                           # leave the output at zero

`daq.ao.channels.dc()` is a **broadcast read** over the `ChannelList`: it
returns the parameter for all four pins at once, rather than asking each in
turn. The same works for any parameter the channels share.

In [ ]:
daq.ao.channels.dc()

## 2.2 Waveform

### Waveform settings live on `daq.ao`, not on the pin

The board has **one** AO clock, so shape, frequency, amplitude and update rate
belong to the subsystem. Only *which pin carries it* is per-pin.

| parameter | meaning |
|---|---|
| `daq.ao.shape` | `"square"`, `"sine"`, `"triangle"`, `"ramp"` |
| `daq.ao.freq` | Hz — snapped so a whole number of samples fits one period |
| `daq.ao.amp` | volts, peak |
| `daq.ao.offset` | volts |
| `daq.ao.rate` | DAC update rate, S/s — sets edge resolution and buffer length |
| `daq.ao.duty` | square duty cycle |
| `daq.ao.actual_freq` | what you really got after snapping |
| `daq.ao.onboard` | **True if the waveform loops from the board's own memory** |

`onboard` matters for long runs: if the buffer fits the onboard FIFO the loop
needs no USB traffic and a saturated input stream cannot starve it. If it does
not fit, the waveform is pushed across USB continuously and will eventually
underflow (`-200621`).

In [ ]:
# Section 2.2 is untriggered.
# Name it "off" all the same: ao.setup is not exclusive, so anything already
# playing a role would still be in `active` and the unpacking would surprise.
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100)
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=5)

print("asked for  ", daq.ao.freq(), "Hz")
print("buffer     ", f"{daq.ao.buffer_len():,} samples of {daq.ao.fifo_samples():,} FIFO")
print("onboard?   ", daq.ao.onboard(), " <- must be True for a long run")

### `with daq.ao.generating():` — generate for the duration of a block

```python
with daq.ao.generating(**overrides):
    data = daq.ai.acquire()
```

Equivalent to `daq.ao.start()` … `daq.ao.stop()`, but the stop is **guaranteed
even if the measurement raises** — which matters, because a buffered output
otherwise holds its last value forever.

Any `ao` parameter can be overridden for the block and is restored afterwards:

```python
with daq.ao.generating(freq=200, amp=0.5):
    ...
```

In [ ]:
with daq.ao.generating():
    data = daq.ai.acquire()

print(f"{len(data['ai0']):,} samples, {data['ai0'].min():+.3f} .. {data['ai0'].max():+.3f} V")

### `trace` and `time_axis` — the QCoDeS view of the same data

* `ai0.trace()` — **never measures**. The hardware produces every enabled
  channel in one block, and `acquire()` split it into rows; `trace` just picks
  this channel's row. It **raises** if there is no scan in hand, rather than
  handing back a stale array.
* `daq.ai.time_axis()` — sample index ÷ *actual* rate. The clock divides a
  20 MHz timebase, so the rate the hardware programmed is rarely exactly the
  one you asked for, and only the real one gives a correct axis.

So there is exactly one line in your code where the hardware runs, and it is
one you wrote — `daq.ai.acquire()`.

In [ ]:
y = ai0.trace()                  # picks ai0's row out of the last acquisition
t = daq.ai.time_axis()           # seconds, from the ACTUAL sample rate

print(f"requested rate {daq.ai.rate():,.0f} S/s")
print(f"actual rate    {daq.ai.actual_rate():,.3f} S/s   <- use this one")

In [ ]:
start, end = 0, 1000

plt.plot(t[start:end] * 1e3, y[start:end])
plt.xlabel("t [ms]"); plt.ylabel("V [V]")
plt.grid(alpha=0.3)

### `reset()` — a clean slate between sections

Roles **accumulate**. `ao.setup` only touches the channels you name, so once
`ao1` is a DC line it stays one, and a later `setup({"ao0": ...})` returns
**two** channels — which is why `ao0, = ...` raises `too many values to unpack`
part way down a notebook.

| call | clears |
|---|---|
| `daq.ao.reset()` | every role back to `"off"`, pins written to **0 V first** |
| `daq.ai.reset()` | all channels disabled, `trigger` cleared, `on_armed` forgotten |
| `daq.reset()` | both, and stops the counters |

`zero=True` (the default) writes 0 V **before** switching a pin off, because
`"off"` does not mean 0 V — an untimed output holds its last value, so
switching a 5 V trigger line off would leave 5 V on the screw.

What `reset()` deliberately does *not* touch: `pps.terminal` and
`marker.terminal`. Those describe the **wiring**, not the configuration — the
cable has not moved.

`daq.ai.reset()` matters for a second reason: `set_on_armed` persists. A
callback registered for one acquisition fires on **every** acquisition
afterwards, which is what you want inside one section and a surprise in the
next.

In [ ]:
daq.reset()          # roles off and pins at 0 V, inputs disabled,
                     # trigger and on_armed cleared. Wiring untouched.

print("ao roles :", [c.role() for c in daq.ao.channels])
print("ao levels:", daq.ao.channels.dc())
print("ai active:", [c.short_name for c in daq.ai.active])
print("ai trigger:", daq.ai.trigger())

In [ ]:
y = ai0.trace()                  # picks ai0's row out of the last acquisition
t = daq.ai.time_axis()           # seconds, from the ACTUAL sample rate

print(f"requested rate {daq.ai.rate():,.0f} S/s")
print(f"actual rate    {daq.ai.actual_rate():,.3f} S/s   <- use this one")

In [ ]:
start, end = 0, 1000

plt.plot(t[start:end] * 1e3, y[start:end])
plt.xlabel("t [ms]"); plt.ylabel("V [V]")
plt.grid(alpha=0.3)

## 2.3 Waveform, triggered

A **start trigger** makes a task wait for an edge on a terminal instead of
starting immediately. Point `ai.trigger` and `ao.trigger` at the same line and
both start on one edge, so the waveform has a known phase at sample 0.

Here both triggers are on **PFI 8**, while the 1 pps arrives separately on
**PFI 9**. So the acquisition starts when the marker fires, and the atomic
seconds are recorded independently — sample 0 is *not* an atomic second, and
the first edge lands somewhere in the first second.

That is the general case and the one worth learning. Putting both on the same
pin is a special case covered below.

### The trigger comes from the external square wave generator on 

**PFI 8 (terminal 81) is physically connected to an external square wave generator.**
The generator provides digital trigger pulses/edges directly on .

Because the generator is external, both  and  can trigger on  directly:



**No  callback or  pulse needed.** The external square wave generator on  triggers the acquisition automatically as soon as the tasks arm on the rising edge.

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100, trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=5, trigger="/Dev1/PFI8")

print("ao.trigger:", daq.ao.trigger(), " ai.trigger:", daq.ai.trigger())

In [ ]:
with daq.ao.generating():
    data = daq.ai.acquire()    # arms and triggers on rising edge from external generator on PFI 8

---
# 2. Clock check — the board against the rubidium

The board's clock does not know it is wrong. It divides its 20 MHz timebase,
reports a sample rate with confidence, and drifts by seconds over a day.

The FS725 emits one pulse per **atomic** second. Comparing them is what makes a
multi-day time axis mean anything.

### How the counter is used, and why it looks backwards

| | |
|---|---|
| what is **counted** | `ai/SampleClock` — the board's own sample clock |
| what **latches** it | the 1 pps, used as the counter task's *sample clock* |

So each value the counter returns is *the scan index that was current when an
atomic second arrived*. That is the edge table — complete, edge-triggered,
nothing missed, in the same units as the data.

Feeding the 1 pps into an analog channel instead would miss ~75 % of the pulses
at 25 kS/s, because the ADC samples instantaneously and the pulse is 10 µs wide.

### `daq.clock_check(seconds=None, verbose=True)`

Acquires for `seconds`, then reports the true rate. If a pin has role
`"wave"`, the waveform runs during the measurement and is measured back as a
cross-check — but note it can only ever **confirm**, never detect: the DAC and
the ADC divide the same timebase, so they are wrong together.

**How long to run it** — each edge is resolved to ±1 scan (40 µs at 25 kS/s),
and the estimate spans the whole measurement:

| `seconds=` | resolution | equivalent |
|---|---|---|
| 10 | ~6 ppm | ±0.5 s/day |
| 60 | ~1 ppm | ±0.09 s/day |
| 600 | ~0.1 ppm | ±0.01 s/day |

Use 10 s to confirm the wiring. Use 600 s once, for the number you would quote.

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100, trigger="/Dev1/PFI8")

ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=5, trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI9")
daq.pps.counter("ctr1")

daq.check()

In [ ]:
result = daq.clock_check(seconds=10)

### Reading the output

* **edges caught** should be about `seconds`. Far fewer means the pulse is not
  reaching the terminal — check `daq.wiring()`.
* **discarded values** are normal: every acquisition ends with one latched
  value that is *not* an atomic second (when the AI task stops the sample clock
  stops, the count freezes, and one more value comes back carrying it).
* **ppm** in the tens is normal — the board is specified to ±50 ppm. What
  matters is that it is *stable*, because that is what makes it correctable.
* **the square-wave line**, if present, should agree with the rubidium.

---
# 3. The atomic time axis

### `daq.acquire_with_pps()`

Returns `(data, edges)` — one record on every enabled channel, plus the atomic
seconds that fell inside it, as **scan indices**. An event at scan *k* and an
atomic second at scan *e* are therefore on the same ruler: no clock conversion,
no assumption about the rate.

It lives on the instrument, not on a submodule, because it drives two: the
counter has to start **before** the AI clock runs, and be stopped and joined
before its queue is drained.

### A time axis is a SCALE and a ZERO, and they come from different places

This is the one idea in this section. Keep them apart and everything else
follows.

| | comes from | |
|---|---|---|
| the **scale** | the rubidium | consecutive 1 pps edges are one atomic second apart *by definition*. The edge table **is** the ruler — no sample rate is assumed anywhere, which is what absorbs the drift |
| the **zero** | you | it is a convention. The default is **scan 0**, the first sample of the record — on a triggered run, the instant the trigger fired |

Why scan 0 and not the first 1 pps edge? Because on a triggered run the first
edge is wherever the *phase* between the rubidium and the trigger happens to
put it. In the run below it lands at scan 20,698 — 0.83 s in — and that 0.83 s
is not a measurement of anything. Zeroing there would push every timestamp in
the record by an arbitrary amount.

### Two time axes

| | |
|---|---|
| `daq.ai.time_axis()` | scan index ÷ actual rate — the **board's** clock |
| `daq.ai.atom_time_axis()` | interpolated between 1 pps edges — the **rubidium** |

**Same zero, different scale.** The second one never uses the sample rate: it
asks *where does this sample sit between two atomic ticks?* Because they share
a zero, subtracting them gives you the drift and nothing else.

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100, trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=60 * 3, trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI9")
daq.pps.counter("ctr1")

daq.check(strict=True)

In [ ]:
with daq.ao.generating():      # this context manager starts and stops the generator
    data, edges = daq.acquire_with_pps()


In [ ]:
t_board  = daq.ai.time_axis()        # the board's own clock
t_atomic = daq.ai.atom_time_axis()   # corrected against the rubidium

print(f"board  {t_board[0]:+.4f} .. {t_board[-1]:,.4f} s")
print(f"atomic {t_atomic[0]:+.4f} .. {t_atomic[-1]:,.4f} s")
print(f"difference at the end: {(t_atomic[-1] - t_board[-1]) * 1e3:+.3f} ms")

### The drift plot

`t_atomic - t_board` is the board's clock walking away from the rubidium. The
slope is the error in ppm; the **staircase** is real, not noise: each edge is
quantised to one scan (40 µs) and the correction is linear between edges, so
each step lasts one second.

The dashed lines mark the atomic seconds.

This plot is only readable because the two axes share a zero. It is drawn in
**microseconds**, and the whole signal over 300 s is a few thousand of them —
so any constant offset between the axes, of any size, buries it. That is
exactly what an earlier version of `atom_time_axis()` did: it zeroed on the
first edge, the offset was 828 ms = 828,000 µs, and the drift curve went off
the bottom of the axes as a flat line.

In [ ]:
start, end = 0, len(t_atomic) - 1000
fig, ax = plt.subplots(figsize=(14, 5))

for edge in edges[::10]:                 # every 10th, or the plot is solid black
    if edge > end:
        break
    ax.axvline(t_atomic[edge], color="black", alpha=0.3, lw=1, ls="--")

res = t_atomic[start:end] - t_board[start:end]
ax.plot(t_atomic[start:end], res * 1e6, color="red", lw=2)

ppm = np.polyfit(t_board[start:end], res, 1)[0] * 1e6
ax.set_xlabel("t [s]"); ax.set_ylabel(r"$\Delta t$ [$\mu$s]")
ax.set_title(f"clock error {ppm:+.2f} ppm  ->  {abs(ppm) * 0.0864:.2f} s/day")
ax.grid(alpha=0.3)

### Choosing the zero yourself

`atom_time_axis()` always uses scan 0. The free function underneath it takes an
`origin`, and the three settings are the three questions you might be asking:

| `origin=` | t = 0 at | ask it when |
|---|---|---|
| `"start"` *(default)* | scan 0 — the trigger | "how long after the shot started?" |
| `"edge"` | the first 1 pps edge kept | "where do the atomic ticks fall?" |
| a scan index | that scan | "how long before/after **this** event?" — e.g. `origin=marks[2]` |

Nothing about the measurement changes. It is a subtraction applied after the
interpolation, so **every difference of two times is identical** whichever you
pick — the drift slope, the interval between two markers, the length of the
record. Only the label on zero moves.

In [ ]:
from postprocess import times_from_edges

idx = np.arange(len(t_board))
t_start = times_from_edges(idx, edges, scans_per_second=daq.ai.actual_rate())
t_edge  = times_from_edges(idx, edges, scans_per_second=daq.ai.actual_rate(),
                           origin="edge")

kept, _ = clean_edges(edges, daq.ai.actual_rate())
print(f"first 1 pps edge at scan {kept[0]:,}"
      f"  = {t_start[kept[0]]:.5f} s after the trigger\n")
print(f"origin='start'  {t_start[0]:+.5f} .. {t_start[-1]:.5f} s")
print(f"origin='edge'   {t_edge[0]:+.5f} .. {t_edge[-1]:.5f} s")
print(f"constant offset between them: "
      f"{(t_start - t_edge).mean():.5f} s, "
      f"varying by {np.ptp(t_start - t_edge) * 1e9:.1f} ns")

# the same measurement either way - the span is identical
print(f"\nspan: {t_start[-1] - t_start[0]:.6f} s  vs  "
      f"{t_edge[-1] - t_edge[0]:.6f} s")

### The check that this is right

The atomic axis should agree with the naive one you would build by hand from
`np.diff(edges)` — because that is the same measurement, just with the drift
fitted as a single straight line instead of corrected second by second. If they
disagreed by more than the staircase, one of them would be wrong.

In [ ]:
m = np.diff(kept).mean()          # scans per atomic second, averaged
t_hand = idx / m                  # the by-hand axis: one number for the rate

print(f"{m:,.3f} scans per atomic second  (std {np.diff(kept).std():.3f})")
print(f"max |atom_time_axis - by hand| = "
      f"{np.abs(t_start - t_hand).max() * 1e6:.1f} us over "
      f"{t_start[-1]:.0f} s")
print("\n^ that residual IS the per-second correction: the by-hand axis uses")
print("  one average rate, atom_time_axis follows each atomic second.")

---
# 4. Long run — recording to disk

### `daq.long_run(outdir, ...)`

| argument | meaning |
|---|---|
| `outdir` | a **new** directory. It refuses to reuse one — two runs in a directory cannot be told apart afterwards |
| `hours` | how long. `None` runs until you stop it |
| `rotate_minutes` | `None` (default) = **one file per channel** for the whole run. `60` = a new file every hour |
| `background` | `True` returns a handle immediately and records in a thread |
| `require_pps` | `False` allows a run with no rubidium (bench testing) |
| `live_decimate` | how many scans per summary block for `live_preview` (100) |
| `live_seconds` | how far back that summary reaches (60) |
| `to_hdf5` | also convert closed segments to `.h5` in the background |

`live_decimate` and `live_seconds` **do not affect what is recorded** — only
the little in-RAM buffer you glance at.

### What ends up on disk

```
outdir/
    run.json        the instrument settings at the start
    manifest.jsonl  which file holds which scans — one JSON per line
    edges.i64       every 1 pps edge of the whole run, as scan indices
    markers.i64     every marker, same units — absent if you sent none
    seg_00000_ai0_20260813T101500Z.f32     the samples: raw float32
```

Nothing is filtered before writing. The `.f32` is what DAQmx returned; the
edge table includes its artefacts. **A recording you cannot re-take must not be
stored already interpreted** — if the interpretation turns out to be wrong, the
raw file is what lets you fix it tomorrow instead of measuring again.

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100, trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000,  trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI9")
daq.pps.counter("ctr1")

print(daq.ai.actual_rate(), "S/s actual")

### Check twice before committing hours

`onboard` must be **True**. If it is False the waveform is regenerated across
USB and will underflow after a few hours of competing with the input stream —
a failure that appears in the middle of the night, not now.

In [ ]:
daq.check(strict=True)                       # raises if anything cannot run

print("AO regenerates onboard:", daq.ao.onboard(),
      f"({daq.ao.buffer_len():,} of {daq.ao.fifo_samples():,} FIFO samples)")

rate = daq.ai.actual_rate()
print(f"disk rate: {rate * 4 / 1e6:.2f} MB/s = {rate * 4 * 86400 / 1e9:.1f} GB/day")

### Rehearse for 3 seconds

Look at the signal and the edges before spending hours. A wiring mistake found
here costs three seconds.

`edge_report(edges, nominal_rate)` says whether a run's timing is sound: how
many latched values were discarded and why, how many atomic seconds were
**missed** (kept as gaps — the numbering accounts for them), and the board's
rate and drift measured across the whole record.

In [ ]:
daq.ai.duration(3.0)
data, edges = daq.acquire_with_pps()

print(f"{len(data['ai0']):,} samples, "
      f"{data['ai0'].min():+.3f} .. {data['ai0'].max():+.3f} V\n")
edge_report(edges, nominal_rate=daq.ai.actual_rate())

### Start

AO first, so it is armed before the input task starts. `background=True`
returns immediately — this cell does not block, and the acquisition continues
while you run the cells below.

**Put `OUTDIR` on a local disk.** A sync client (OneDrive, Dropbox) trying to
upload a continuously growing file will stall the write, and a stalled write
becomes a buffer overflow.

In [ ]:
daq.ao.start()                     # arms; fires on the next PFI8 edge

OUTDIR = "long_run_test"           # a NEW directory each time

run = daq.long_run(OUTDIR,
                   hours=0.1,      # 6 minutes
                   rotate_minutes=2,
                   background=True)
run

## 4.1 Watching it — the two paths

There are two places live data exists, and choosing the wrong one is the usual
confusion:

| | `daq.live_preview()` | the files (`LiveScope`, `RunFiles`) |
|---|---|---|
| where | a deque in RAM | disk |
| how much | last `live_seconds` (60) | **everything** |
| channels | the first one only | all of them |
| resolution | min/max per `live_decimate` block | every sample |
| answers | *"is it alive?"* | *"what did it measure?"* |

`run.status()` gives progress: seconds recorded, current segment, and whether
the thread is still alive.

In [ ]:
s = run.status()
print(f"{s['seconds']:,.1f} s recorded, segment {s['segment']}, alive={run.alive}")

t, y = daq.live_preview()
print(f"live buffer: {len(y):,} points spanning {t[-1] - t[0]:.1f} s"
      if len(y) else "live buffer empty")

## 4.2 Live plot

**Two things must happen every frame**, and missing either is why a live plot
appears not to work:

1. **clear the axes** — `ax.plot()` in a loop *adds* a line each time, so after
   a few hundred frames the figure is hundreds of lines and grinds to a halt;
2. **push the figure to the screen** — with the default `%matplotlib inline`
   backend there is no GUI event loop, so `plt.pause()` redraws nothing and the
   cell just blocks. The figure has to be re-`display`ed with the previous
   output cleared.

`time.sleep()` alone never works on inline: it sleeps without redrawing.

> **A note on what you are looking at.** The live buffer stores the **min and
> max** of each block of `live_decimate` samples, not every 100th sample.
> Striding would be undersampling — at 250 effective S/s a 100 Hz square has
> 2.5 samples per period and renders as ragged nonsense with the wrong duty
> cycle, while the file is perfect. Min/max cannot misrepresent the envelope,
> but the *order within a block* is not real. For the true waveform, use §5.3.

In [ ]:
# RECIPE A - works with the DEFAULT inline backend, nothing to install
from IPython.display import display, clear_output

fig, ax = plt.subplots(figsize=(12, 4))

try:
    while run.alive:
        t, y = daq.live_preview()
        ax.clear()                                  # (1) or the lines pile up
        if len(y):
            ax.plot(t, y, lw=1)
            ax.set_xlabel("board time [s]"); ax.set_ylabel("V [V]")
            ax.set_title(f"{run.status()['seconds']:,.0f} s recorded")
            ax.grid(alpha=0.3)
        clear_output(wait=True)                     # (2) wait=True = no flicker
        display(fig)
        time.sleep(0.5)
except KeyboardInterrupt:
    print("stopped plotting (the run continues)")

**Recipe B** is better if you install `ipympl` (`pip install ipympl`): put
`%matplotlib widget` in its own cell above and re-run from there. You get real
pan and zoom, no flicker, and updating one line's data is much cheaper than
redrawing.

In [ ]:
# RECIPE B - needs %matplotlib widget
fig, ax = plt.subplots(figsize=(12, 4))
line, = ax.plot([], [], lw=1)
ax.set_xlabel("board time [s]"); ax.set_ylabel("V [V]"); ax.grid(alpha=0.3)

try:
    while run.alive:
        t, y = daq.live_preview()
        if len(y):
            line.set_data(t, y)          # update the SAME line - no pile-up
            ax.relim(); ax.autoscale_view()
        plt.pause(0.5)                   # redraws AND sleeps
except KeyboardInterrupt:
    pass

## 4.3 An oscilloscope view

What makes a scope readable is not the drawing — it is the **trigger**. Without
it the waveform slides across the screen every frame; with it, it stands still.

This reads **real samples from the file**, not the live buffer, because the
live buffer is an envelope and its intra-block order is not real.

* `RunFiles(outdir, channel)` — the run's files as they grow. Re-reads the
  manifest and file sizes on demand, because a cached size is a lie 200 ms
  later, and finds the **open** segment by name since it has no manifest entry
  until it closes.
* `files.n_scans()` — how much has been written so far.
* `files.read(start_scan, n)` — just that window, one seek and one read, across
  segment joins. A 10 s window of a week-long run touches about a megabyte.

In [ ]:
files = RunFiles(OUTDIR, "ai0")
rate = files.rate

SPAN, PRE = 1500, 100        # samples on screen, samples shown before the trigger

def trigger(y, level=0.0, pre=PRE, n=SPAN):
    """Align on the first rising crossing - a scope trigger, four lines."""
    above = y >= level
    hits = np.flatnonzero(~above[:-1] & above[1:]) + 1
    hits = hits[hits >= pre]                 # need room to show the run-up
    if len(hits) == 0:
        return y[:n]                         # free-run if nothing crosses
    return y[hits[0] - pre : hits[0] - pre + n]

fig, ax = plt.subplots(figsize=(12, 4))
line, = ax.plot([], [], lw=1)
ax.set_xlabel("ms"); ax.set_ylabel("V"); ax.grid(alpha=0.3)

try:
    while run.alive:
        total = files.n_scans()
        y = files.read(max(0, total - 5000), 5000)     # the last 0.2 s
        if len(y) < SPAN:
            time.sleep(0.2); continue
        y = trigger(y)
        line.set_data(np.arange(len(y)) / rate * 1e3, y)
        ax.set_xlim(0, len(y) / rate * 1e3)
        ax.set_ylim(y.min() - 0.1, y.max() + 0.1)
        ax.set_title(f"{total / rate:,.0f} s recorded")
        clear_output(wait=True); display(fig)
        time.sleep(0.2)
except KeyboardInterrupt:
    print("stopped")

## 4.4 `LiveScope` — navigate the whole run while it records

```python
scope = LiveScope(outdir, channel="ai0", width=5.0)
scope.show()          # follows the end of the file; interrupt to stop
scope.goto(120.0)     # jump to t = 120 s, stop following
scope.zoom(0.1)       # 10x narrower
scope.pan(-1)         # one window back
scope.follow()        # back to live
scope.step()          # draw ONE frame, for your own loop
```

It draws **min/max per pixel column**, which is what a digital scope does and
for the same reason: a 300 s window is 7.5 M samples against ~1500 pixels, and
taking every 5000th sample would drop a 200 µs spike about 999 times in 1000.
Measured on a synthetic run with fifteen 200 µs spikes: min/max keeps 15 of 15,
striding keeps **0**. Zoom in past one sample per column and it stops
decimating entirely.

Nothing in `livescope` touches the DAQ, so a slow plot cannot stall the
acquisition — you can even run it in another process.

In [ ]:
scope = LiveScope(OUTDIR, channel="ai0", width=0.05)   # 50 ms: no decimation
scope.show()          # interrupt the kernel to stop

In [ ]:
run.stop()            # closes the current file cleanly
daq.ao.stop()         # waveform off, ao0 back to 0 V
print("alive:", run.alive)

## 4.5 Recording raw ADC codes instead of volts

Everything above wrote **volts** — `float32`, already scaled by DAQmx. The
other format `long_run` can write is the converter's own **codes**, as
little-endian `uint32`, which is what the group we send data to stores.

It is one line, and it belongs to the **channel configuration** rather than to
the run, so there is no way to set the channels up one way and record them
another:

```python
daq.ai.datatype("uint32")
```

| | `float32` (default) | `uint32` |
|---|---|---|
| file | `seg_00000_ai0_….f32` | `seg_00000_ai0_….u32` |
| holds | volts | converter codes, offset binary |
| bytes per sample | 4 | 4 |
| exactness | rounded to float32 | exactly what the ADC produced |

**The size is identical.** This is about the format being one another group can
read, not about saving space.

### What makes codes readable later

A code means nothing without the range it spans and the bits it spans it in, so
`long_run` writes both into the manifest header — the range **read back from
the hardware**, because `min_val`/`max_val` only *request* a range and DAQmx
snaps it to a real one, and the codes belong to the snapped one:

```
volts = v_min + code * (v_max - v_min) / (2**bits - 1)
```

That is everything the other end needs: `np.fromfile(path, dtype="<u4")` and
three numbers. On this side `load_run` applies it for you.

In [ ]:
# The same long_run as above, with ONE line different. Kept deliberately
# minimal: no trigger, so it starts immediately, and require_pps=False so it
# runs whether or not the rubidium is connected.
daq.reset()

ai0, = daq.ai.setup({"ai0": (-1.5, 1.5)}, rate=25_000)
daq.ai.datatype("uint32")                     # <- the only difference
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100)

daq.check(strict=True)
daq.ao.start()

OUTDIR_U32 = Path(time.strftime("codes_run_%H%M%S"))
run = daq.long_run(OUTDIR_U32, hours=30 / 3600, require_pps=False)
daq.ao.stop()

print(sorted(p.name for p in OUTDIR_U32.glob("seg_*")))

### Read it back, both ways

`units="volts"` is the default and behaves exactly as it did for the `.f32`
runs above — the fact that the file holds codes is not something the rest of
your analysis has to know. `units="codes"` gives you the integers as stored.

In [ ]:
manifest, edges = load_long_run(OUTDIR_U32)
header = run_header(manifest)
scaling = header["scaling"]["ai0"]

print(f"datatype on disk : {header['datatype']}")
print(f"range read back  : {scaling['v_min_actual']:+g} .. "
      f"{scaling['v_max_actual']:+g} V over {scaling['bits']} bits")

idx, codes = load_run(OUTDIR_U32, "ai0", units="codes")
idx, volts = load_run(OUTDIR_U32, "ai0", units="volts")
print(f"\n{len(codes):,} samples")
print(f"  codes  {codes.min():>9,} .. {codes.max():>9,}   ({codes.dtype})")
print(f"  volts  {volts.min():+9.4f} .. {volts.max():+9.4f} V ({volts.dtype})")

# And what the OTHER GROUP does: bare numpy, the file, and three numbers -
# nothing from this driver.
span = (1 << scaling["bits"]) - 1
raw = np.fromfile(sorted(OUTDIR_U32.glob("*.u32"))[0], dtype="<u4")
theirs = (scaling["v_min_actual"] + raw
          * (scaling["v_max_actual"] - scaling["v_min_actual"]) / span)

print(f"\nreconstructed from the file with numpy alone: "
      f"max difference {np.abs(theirs - volts[:len(raw)]).max():.2e} V")

---
# 5. Markers — recording *when* things happened

The 1 pps gives you a **ruler**: every edge is one atomic second, and you know
the scan index of each. What it does not tell you is where the interesting
moments are — when the OPX started a run, stopped one, or paused to drain its
buffer.

`daq.marker` is the other half. It is the **same counter trick** as the 1 pps,
on the free counter, pointed at a different wire:

| wire | counter | what you get |
|---|---|---|
| rubidium 1 pps | `ctr1` (`daq.pps`) | atomic seconds — the ruler |
| OPX marker line | `ctr0` (`daq.marker`) | events — when things happened |

Both record **scan indices**, and that is the whole point: markers, 1 pps
edges, samples and segment boundaries are all the same kind of number. Relating
an event to the data is `y[mark]`; relating it to the rubidium is one call.
Nothing is converted, so nothing can be converted wrong.

**Why the scan index and not a time.** A time would have to be derived by
dividing by a rate that is itself wrong by ~12 ppm, and frozen into the file at
whatever the driver's clock correction happened to be that day. A scan index is
what the hardware measures, it is an exact integer, it indexes the data
directly, and it can be **re-timed** when the correction improves — which it
has, twice.

**One line, one kind of pulse.** Only one counter is free, so the hardware
cannot tell "start" from "stop". Let **order** carry the meaning: your OPX
program defines the sequence, so mark 0 is whatever it emits first. The DAQ
only has to say *when*.

**What it buys.** The DAQ records continuously; the OPX does not. The markers
say which stretches of an unbroken recording correspond to the OPX doing
something — and which are the gaps while it reset or drained. The DAQ becomes
a witness for an instrument that stops, because it never does.

## 5.1 A bench trial — the board triggers and marks itself

Before wiring the OPX, simulate it with a **square wave generator**: one line
doing exactly what the OPX will do — an edge that starts the run, then more
edges to mark what happens during it.

This is a **smoke test**: does a marker come back at all, and does it land
where the rubidium says it should. Section 5.2 does the thorough version — a
long run, several segments, and the phase of the generator against the 1 pps
followed all the way through it.

### Three signals, three PFI lines — but only **two** counters

That is not a shortage, because the three do different jobs:

| signal | what it needs | counter |
|---|---|---|
| **1 pps** → `daq.pps` | a counter to **timestamp** each pulse | `ctr1` |
| **markers** → `daq.marker` | a counter to **timestamp** each pulse | `ctr0` |
| **trigger** → `ai.trigger` | the timing engine's start-trigger input | **none** |

A start trigger is not recorded — it **defines scan 0**. There is nothing to
timestamp, because the answer is always zero. So two counters is exactly
enough, and both are spoken for.

### One line does both — the trigger IS the first marker

A PFI input fans out inside the board, so **one wire** can feed the start
trigger *and* a counter at the same time:

```python
daq.ai.setup(..., trigger="/Dev1/PFI8")
daq.marker.terminal("/Dev1/PFI8")        # the SAME line
```

Then the pulse that starts the acquisition is also its first marker, and it
lands at **scan 0** — which is where the trigger is by definition:

```
marks[0] == 0        the trigger
marks[1:]            whatever the OPX did next, on the same line
```

One cable, one PFI line, and the trigger arrives already timestamped on the
same ruler as everything else. `daq.check()` prints a **note** when it sees
this arrangement, so nobody later mistakes that zero for rubbish.

**That clean picture assumes ONE deliberate pulse**, emitted from
`ai.set_on_armed()` — which is how the OPX will do it. A **free-running
generator** behaves differently, and the difference is not subtle: see *The
leading zeros* below before you read `markers.i64`.

### ⚠ The marker counter's resolution is ONE SCAN — and it must not be flooded

Two limits. The first one decides what this trial can measure at all.

**Resolution.** The counter latches the **AI scan count**, so a marker is a scan
index and nothing finer: at 25 kS/s that is **40 µs**, and a single edge is
placed to about ±20 µs. Two pulses inside one scan period latch the *same*
value — which `_marker.py` reports rather than filters, because "two pulses in
one scan" is real information. Point this at a **200 kHz** square (5 µs period)
and **eight consecutive edges share one scan index**: the marker table becomes
groups of eight identical numbers, and where that generator sits inside the
atomic second is simply not recoverable. The *mean* interval still returns
200 kHz — the span and the count are both exact — but no individual edge is
placed better than one scan.

**Throughput.** Every rising edge also costs a blocking DAQmx read in a Python
thread (`_stamper.py`, `drain()`), a queue entry, and 8 bytes on disk. At
200 kHz that is 200,000 reads/s and **1.6 MB/s of `markers.i64`** — sixteen
times the 100 kB/s of the AI channel it is annotating. The counter task's
buffer overflows within milliseconds, `drain()` starts erroring, and after
`MAX_DRAIN_ERRORS` (30, a second apart) it **gives up for the rest of the
run**: markers stop, and the acquisition carries on without them.

So pick the row that matches what you want:

| what you want | how |
|---|---|
| the generator's edges timestamped, and its **phase and drift against the rubidium** | **set the generator to a few Hz.** Every edge is then resolvable and every one is a useful marker — this is what 5.1 and 5.2 do |
| to keep the generator at **200 kHz** | use it as the **trigger only** — leave `daq.marker.terminal(None)`. `trigger_time(OUTDIR)['first_edge_seconds']` still says where scan 0 fell relative to the first atomic second, to the same 40 µs, so the marker would have added nothing |
| a 200 kHz phase measured **properly** | not with this mechanism. That wants a counter doing period measurement against the board's 80 MHz timebase (12.5 ns). `ScanStamper` deliberately counts the *AI sample clock* instead, because its job is to put events on the **data's** ruler, not to be a time-interval counter |

### ⚠ Never drive a PFI input negative

PFI lines are **TTL inputs**, absolute maximum about **−0.5 V to +5.5 V**. The
±1 V square used earlier in this notebook would put −1 V on the pin, which is
outside that. Set the generator to swing **0 V to 5 V** (or 0 to 3.3 V), never
below ground.

### Wiring

| from | pin | to | pin | why |
|---|---|---|---|---|
| **square generator**, 0–5 V, a few Hz | — | **PFI 8** | 81 | trigger **and** markers |
| **AO 0** | 15 | **AI 0** | 1 | loopback: a waveform to look at while the markers are counted |
| rubidium 1 pps | — | **PFI 9** | 83 | the ruler |

Returns: AO GND **16**, AI GND **3**, D GND **82** (each PFI 8–15 sits next to
a D GND pin).

`AO 0` → `AI 0` is the loopback: the DAC generates a waveform and the ADC reads
it back, so the recording has real signal in it. `PFI 8` is the trigger *and*
the marker line — marker 0 is the trigger. `PFI 9` is the ruler, where the
1 pps is counted.

In [ ]:
# ONE input for this smoke test: ai0 is the AO loopback, so there is a real
# waveform in the recording. The trigger AND the markers both come from the
# external square generator on PFI 8 - see the wiring table above.
TRIAL_SECONDS = 40.0        # how much DATA to record (counted from the trigger)
MARKER_HZ = 5.0             # what the square generator is set to. A FEW Hz -
                            # see the rate warning above before raising it.

ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100, trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-1.5, 1.5)}, rate=25_000, duration=TRIAL_SECONDS, trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI9")       # the ruler
daq.marker.terminal("/Dev1/PFI8")    # the SAME line as the trigger
daq.pps.counter("ctr1"); daq.marker.counter("ctr0")

daq.check(strict=True)               # prints a NOTE about the shared line
daq.terminal_roles()

In [ ]:
daq.wiring()

# and the two channels are not sampled at the same instant - one ADC, one mux
for ch in daq.ai.active:
    print(f"{ch.short_name}: sampled {ch.time_offset()*1e6:5.2f} us into each scan")

### Run it

The order matters and mirrors the real thing:

1. the generator is already running on PFI 8, swinging 0 → 5 V
2. **`daq.ao.start()`** arms the waveform. Without it the AO task is configured
   and never started, `ao0` holds 0 V, and the loopback records nothing but the
   ADC's own noise floor — a few hundred microvolts of nothing. The markers
   still work, which is exactly what makes this easy to miss
3. `long_run` starts. Both counters begin immediately, but the **AI task waits**
   for PFI 8 — so every 1 pps edge during the wait latches 0, and `clean_edges`
   discards them on read-back
4. the next rising edge on PFI 8 is the **trigger**: that instant is scan 0
5. every later edge of the same square is an ordinary marker

### The leading zeros are NOT one trigger — they are every edge that arrived early

This is the one thing a free-running generator does that a single deliberate
pulse does not, and it is worth understanding before it looks like a fault.

Building and arming an AI task over USB takes **tens to hundreds of
milliseconds**, and the marker counter is started *before* it
(`long_run` → `marker.start()`, then `pps.start()`, then the AI task). During
that whole window the AI sample clock is not running, so its count is 0 — and
**every generator edge in that window latches 0**:

```
markers.i64 = [0, 0, 0, ..., 0,   25, 50, 75, 100, ...]
               ╰── the arm window ──╯  ╰─── real markers ───╯
```

At 1 kHz an 80 ms arm window puts about **80 zeros** at the front. The
triggering edge is in there too, and it is *indistinguishable* from the rest,
because it latched the count at the instant the clock started — which is still
0. So, concretely:

* `marks[0] == 0` here does **not** mean "the trigger". It means "an edge
  arrived before the clock ran". Only with a single pulse from
  `ai.set_on_armed()` is that leading zero the trigger and nothing else.
* the zeros are kept rather than filtered, because *"arrived while nothing was
  being recorded"* is real information — this is the same thing `_marker.py`
  reports as repeated values.
* **drop them before you measure anything.** Eighty zero-length intervals in a
  mean period is a 0.2% error at 1 kHz, and a phase computed through them is
  meaningless.

`scan 0` itself is still exactly the trigger instant, so the *time axis* is
unaffected. It is only the marker table's first N entries that need reading for
what they are.

In [ ]:
# A new directory per run: long_run REFUSES to reuse one, because two runs in
# one directory cannot be told apart afterwards. The timestamp makes re-running
# this cell painless.
OUTDIR = Path(time.strftime("marker_trial_%H%M%S"))

# AO FIRST, and this line is NOT optional: without it the waveform task is
# configured but never started, ao0 holds 0 V, and ai0 records its own noise
# floor for the whole run. The markers work either way, so nothing complains.
daq.ao.start()                     # arms; fires on the next PFI8 edge

run = daq.long_run(OUTDIR, hours=TRIAL_SECONDS / 3600, background=True,
                   verbose=False)
run.join()
daq.ao.stop()                      # waveform off, ao0 back to 0 V

# Did we record a signal, or a flat line? Check now, not three cells later.
idx_check, y_check = load_run(OUTDIR, "ai0")
span_mv = (y_check.max() - y_check.min()) * 1e3
print(f"done -> {OUTDIR}")
print(f"{len(idx_check):,} scans of ai0, "
      f"{y_check.min():+.4f} .. {y_check.max():+.4f} V  (span {span_mv:,.2f} mV)")
if span_mv < 50:
    print("   !! that is the ADC noise floor, not the loopback. The .f32 is "
          "full of samples, they are just all ~0 V. Check that "
          "daq.ao.start() ran above, and that AO 0 (pin 15) is wired to "
          "AI 0 (pin 1) with AO GND (16) to AI GND (3)")
print(f"{len(load_markers(OUTDIR)):,} markers, "
      f"{len(load_long_run(OUTDIR)[1]):,} latched 1 pps values")

### Read it back — and check the markers against the rubidium

`marker_times(OUTDIR)` gives the scan index **and** the atomic second of every
marker, both from the same edge table that times the samples — so a marker and
the data around it cannot disagree.

With a steady square on the marker line there is a check built in: consecutive
markers are one generator period apart, measured **against the rubidium**, with
no board clock anywhere in the sum. If the generator is trustworthy the
intervals say so; if the rubidium is, they measure the generator.

Expect a **run of zeros** at the front, not one — see *The leading zeros*
above. The cell below counts them and splits them off as `marks_real`, which is
what every measurement then uses. The intervals of `marks_real` are genuine
periods of the generator.

What this does **not** give you is a second opinion on each marker: only
`ai0` is connected, and it carries the AO loopback, not the PFI 8 line. The
cross-check available here is the one above — the intervals, against the
rubidium. 5.2 turns that into the measurement it is really for: the **phase**
between the generator and the 1 pps, over a long run.

In [ ]:
manifest, edges = load_long_run(OUTDIR)
idx, y = load_run(OUTDIR, "ai0")

rate = manifest[0]["nominal_rate"]
t_atomic = times_from_edges(idx, edges, scans_per_second=rate)

print("=== timing ===")
edge_report(edges, nominal_rate=rate)
info = trigger_time(OUTDIR)
if info["first_edge_seconds"] is not None:
    print(f"\ntrigger fired {info['first_edge_seconds']:.4f} s before the "
          f"first atomic second")

print("\n=== markers ===")
marks, t_marks = marker_times(OUTDIR)

# THE LEADING ZEROS. Every edge that arrived while the AI task was still
# building and arming latched 0, because the sample clock had not ticked yet.
# The triggering edge is among them and cannot be told apart. Split them off:
# they are information ("before the clock ran"), but they are not timestamps.
n_zeros = int((marks == 0).sum())
marks_real, t_real = marks[n_zeros:], t_marks[n_zeros:]

print(f"{len(marks):,} markers on {daq.marker.terminal()}")
if len(marks_real) > 1:
    # the arm window, inferred: n_zeros edges at the generator's own period
    period_scans = float(np.median(np.diff(marks_real)))
    print(f"   {n_zeros:,} at scan 0 = the trigger plus every edge that "
          f"arrived while the task was arming,")
    print(f"      about {n_zeros * period_scans / rate * 1e3:,.0f} ms at this "
          f"generator's {rate / period_scans:,.0f} Hz")
else:
    print(f"   {n_zeros:,} at scan 0")
print(f"   {len(marks_real):,} usable markers")

# A steady square gives thousands of them - print the two ends of the real ones.
SHOW = 6
shown = sorted(set(range(min(SHOW, len(marks_real))))
               | set(range(max(0, len(marks_real) - SHOW), len(marks_real))))
previous = None
for i in shown:
    if previous is not None and i != previous + 1:
        print(f"   ... {i - previous - 1:,} more ...")
    print(f"   marker {i:>7}: scan {marks_real[i]:>12,}   "
          f"atomic {t_real[i]:+12.6f} s")
    previous = i

if not len(marks_real):
    print("\n   NO USABLE MARKERS - see the outcome table below")

In [ ]:
# The first 40 ms: the loopback waveform, with every marker that falls inside
# it. These are two DIFFERENT signals - ai0 is the AO loopback, the markers are
# generator edges on PFI 8 - and the point is that they share one axis without
# anything being converted. That is what recording scan indices buys.
fig, ax = plt.subplots(figsize=(12, 4))

start, end = 0, 1000                     # scans, so 40 ms at 25 kS/s
ax.plot(t_atomic[start:end] * 1e3, y[start:end], lw=1, label="ai0 (loopback)")

# the pre-arm pile at scan 0, drawn ONCE rather than as n_zeros overlapping
# lines - otherwise it looks like a solid red band and hides what it is
if n_zeros:
    ax.axvline(0.0, color="darkred", lw=2,
               label=f"scan 0: trigger + {n_zeros - 1:,} pre-arm edges")

inside = (marks_real >= idx[0] + start) & (marks_real < idx[0] + end)
for n, t in enumerate(t_real[inside]):
    ax.axvline(t * 1e3, color="red", alpha=0.5, lw=1, ls="--",
               label="markers" if n == 0 else None)

ax.set_xlabel("atomic ms since the trigger")
ax.set_ylabel("Voltage [V]")
ax.set_title(f"{inside.sum():,} of {len(marks_real):,} usable markers land in "
             f"the first {(end - start) / rate * 1e3:.0f} ms")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")

In [ ]:
# Every rising edge on PFI 8 is a marker, so the marker table measures the
# GENERATOR against the RUBIDIUM - the board's own clock is not in this sum.
# The pre-arm zeros are excluded; they are not timestamps.
intervals = np.diff(t_real)
scans_per_period = float(np.median(np.diff(marks_real)))
f_meas = 1 / intervals.mean()

print(f"{len(marks_real):,} usable markers ({n_zeros:,} zeros dropped)")
print(f"median spacing {scans_per_period:,.1f} scans = "
      f"{scans_per_period / rate * 1e6:,.1f} us")
print(f"measured against the rubidium: {f_meas:,.4f} Hz "
      f"(period {intervals.mean() * 1e6:,.2f} us)")
print(f"MARKER_HZ says {MARKER_HZ:g} Hz -> "
      f"{(f_meas - MARKER_HZ) / MARKER_HZ * 1e6:+,.0f} ppm")
if abs(f_meas - MARKER_HZ) > 0.05 * MARKER_HZ:
    print(f"   !! MARKER_HZ does not match the generator. Set "
          f"MARKER_HZ = {f_meas:,.0f} in the config cell and re-run this one - "
          f"the ppm above is meaningless until you do")

print(f"\ninterval spread: {intervals.std() * 1e6:,.1f} us std, "
      f"{(intervals.max() - intervals.min()) * 1e6:,.1f} us peak to peak")
print(f"one scan is {1e6 / rate:.1f} us and the period is "
      f"{scans_per_period:,.1f} scans -> "
      + ("each edge is individually resolved"
         if scans_per_period >= 4 else
         "TOO FAST: edges are sharing scan indices, so no single edge is "
         "placed better than one scan"))
repeats = int((np.diff(marks_real) == 0).sum())
if repeats:
    print(f"   {repeats:,} pairs of markers share a scan index")

### What each outcome tells you

| result | meaning |
|---|---|
| a run of zeros, then one period per interval | the whole chain works — go on to 5.2, then wire the OPX in |
| **`ai0` spans a fraction of a mV** | the loopback is not running: `daq.ao.start()` was skipped, or AO 0 (pin 15) is not wired to AI 0 (pin 1). The `.f32` is not empty — it is full of the ADC's noise floor |
| **dozens of markers at scan 0** | expected with a free-running generator: that is the arm window, not a fault. One zero only happens with a single pulse from `set_on_armed()` |
| **no marks at all** | `daq.marker.terminal` unset, or the counter never claimed — check `daq.terminal_roles()` |
| far more marks than edges of the generator | the square is bouncing, or its swing barely crosses the 2.5 V PFI threshold |
| far fewer | the pulse never reached the threshold — check the **D GND** return (pin 82) |
| marks stop partway through the run | the marker rate is too high: `drain()` hit `MAX_DRAIN_ERRORS` and gave up. Slow the generator |
| intervals off by a whole period | one edge was missed; the *others* are still exact, because each is timed independently |
| `t_marks` all `+0.000` | only one usable 1 pps edge — no ruler. Check PFI 9 |
| `edge_report` discards nearly everything | the 1 pps is not on PFI 9, or `pps.counter` collides with `marker.counter` |

Once this passes, replacing the generator with the OPX's own output changes
**nothing** in the analysis — same scan indices, same edge table, same two
lines of read-back. The only thing that changes is *what* is emitting the
pulse.

And in the real setup, launch the QM job from `daq.ai.set_on_armed(...)`
rather than a `time.sleep`: a pulse sent before the task arms triggers
nothing and still leaves a stray 0 in the marker table.

---
## 5.2 The real thing — a long run with markers, rotation and HDF5

5.1 proved a marker comes back. This is the run you would actually leave
overnight, with the three things a long recording needs turned on at once:

| | what it adds | why it is off by default |
|---|---|---|
| `rotate_minutes=5` | a **new file every 5 minutes**, so a closed one can be archived while the run continues | for a short run, four files are easier to think about than fifty |
| `to_hdf5=True` | each closed segment is compressed to `.h5` **in a background thread**, metadata attached | costs CPU, and needs `h5py` |
| `marker.terminal` | events, on the same ruler as the samples | not every run has anything to mark |

### The invariant that makes this safe

**`edges.i64` and `markers.i64` are per-RUN, not per-segment.** Both hold
*global scan indices*, so rotation cannot split them and cannot renumber them:

```
scan 0 ─────────────────────────────────────────────────────────► scan N
   │ seg 00000 │ seg 00001 │ seg 00002 │ seg 00003 │      .f32, one per channel
   ╰───────────┴───────────┴───────────┴───────────╯
   ▲     ▲          ▲            ▲          ▲   ▲          markers.i64  (one run)
   │  ▲     ▲    ▲     ▲     ▲     ▲    ▲     ▲   ▲        edges.i64    (one run)
```

Finding which file holds a marker is a manifest lookup, not a conversion:

```python
start_sample <= mark < start_sample + n
```

That is the whole reason the driver records scan indices instead of times. A
marker two hours into a run needs no rate, no offset and no knowledge of how
many times the file rotated.

### What the markers are FOR here — the phase you cannot know in advance

The rubidium's 1 pps and the signal generator are **two unrelated clocks**, and
the phase between them is whatever it happens to be when you switch on. There
is no way to know it beforehand, and it does not stay put — the generator drifts
against the rubidium. The marker counter is how you measure it, because it puts
both on one ruler:

* the **trigger** is a generator edge, and it defines **scan 0**
* every later generator edge is a **marker**, at a known scan index
* the **1 pps** edges are at known scan indices too

So the fractional part of a marker's atomic time is *where that generator edge
sat inside its atomic second*, and watching it across a long run measures the
generator against the rubidium. That is **check 3** below, and it is the reason
to run this for twenty minutes rather than forty seconds: the phase walk needs
time to show up.

**At one scan of resolution, and no better** — see the resolution note in 5.1
before pointing this at a fast generator.

### Wiring — exactly the same as 5.1

| from | pin | to | pin | why |
|---|---|---|---|---|
| **square generator**, 0–5 V, a few Hz | — | **PFI 8** | 81 | trigger **and** markers |
| **AO 0** | 15 | **AI 0** | 1 | loopback: the DAC emits the wave, the ADC reads it back |
| rubidium 1 pps | — | **PFI 9** | 83 | the ruler |

Returns: AO GND **16**, AI GND **3**, D GND **82** (each PFI 8–15 sits next to
a D GND pin). Nothing else needs to be connected.

In [ ]:
# ONE input, because ai0 is the only one wired: it carries the AO 0 loopback,
# so the recording has real signal in it. Everything below is channel-agnostic
# (it reads channels_of(OUTDIR)), so adding channels only multiplies the number
# of .f32 files per segment.
MINUTES = 20.0              # of DATA, counted from the trigger
ROTATE_MINUTES = 5.0        # -> 4 segments
MARKER_HZ = 5.0             # the generator. A FEW Hz - see 5.1's resolution
                            # and rate note before raising it.

ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100,
                    trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-1.5, 1.5)}, rate=25_000, trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI9")       # the ruler
daq.marker.terminal("/Dev1/PFI8")    # the SAME line as the trigger
daq.pps.counter("ctr1"); daq.marker.counter("ctr0")

# HDF5 conversion is optional and fails QUIETLY - _hdf5_worker logs the
# ImportError and returns, so the run keeps going with raw .f32 only. Better to
# find out now than to look for .h5 files in the morning.
try:
    import h5py
    print(f"h5py {h5py.__version__} - segments will be converted as they close")
except ImportError:
    print("!! h5py NOT installed: to_hdf5=True will do nothing. "
          "pip install h5py")

daq.check(strict=True)
daq.wiring()

### Check the budget before committing the time

Two channels doubles the disk rate, and `to_hdf5=True` writes a **second**
copy: gzip on a smooth signal is often 3–5× smaller, but on noise it is barely
smaller than the raw, so size the disk for roughly **2× the raw** and be
pleased when it is less.

`onboard` must be **True**. If it is False the waveform is regenerated across
USB and will underflow after a few hours of competing with the input stream —
a failure that appears in the middle of the night, not now.

**Put `OUTDIR` on a local disk**, not in OneDrive. A sync client uploading a
continuously growing file stalls the write, and a stalled write becomes a
buffer overflow.

In [ ]:
rate = daq.ai.actual_rate()
n_ch = len(daq.ai.active)
n_seg = int(np.ceil(MINUTES / ROTATE_MINUTES))

print("AO regenerates onboard:", daq.ao.onboard(),
      f"({daq.ao.buffer_len():,} of {daq.ao.fifo_samples():,} FIFO samples)")
print(f"{rate:,.3f} S/s x {n_ch} channels")
print(f"raw: {rate * 4 * n_ch / 1e6:.2f} MB/s = "
      f"{rate * 4 * n_ch * 60 * MINUTES / 1e6:,.0f} MB for {MINUTES:g} min")
print(f"     {rate * 4 * n_ch * 86400 / 1e9:.1f} GB/day if you leave it running")
print(f"segments: {n_seg} per channel = {n_seg * n_ch} .f32 files "
      f"(+ as many .h5)")
print(f"markers:  ~{MARKER_HZ * 60 * MINUTES:,.0f} at {MARKER_HZ:g} Hz = "
      f"{MARKER_HZ * 60 * MINUTES * 8 / 1e3:,.1f} kB")
print(f"1 pps:    {60 * MINUTES:,.0f} edges = {60 * MINUTES * 8 / 1e3:,.1f} kB")

### Start

AO first, so the generator is armed before the input task starts.
`background=True` returns immediately — the cells below run while the
acquisition continues in its thread.

`verbose=True` makes the writer announce each rotation and each conversion.
Those prints come from a **background thread**, so Jupyter attaches them to
whichever cell happens to be running — expect them to turn up in the watch
cell below rather than this one.

In [ ]:
daq.ao.start()                     # arms; fires on the next PFI8 edge

OUTDIR = Path(time.strftime("marker_long_run_%H%M%S"))   # a NEW directory

run = daq.long_run(OUTDIR,
                   hours=MINUTES / 60,
                   rotate_minutes=ROTATE_MINUTES,
                   to_hdf5=True,              # convert closed segments as we go
                   background=True,
                   verbose=True)
run

### Watch it — rotation and conversion, live

Three things to watch, and the relationship between them is the point:

* **`.f32` count climbs in steps** of one per channel, every `ROTATE_MINUTES`.
* **`.h5` trails one rotation behind.** Conversion only ever touches a
  **closed** segment — an open file is still being appended to, so it cannot
  be copied without a torn read and cannot be compressed at all.
* **markers and edges climb continuously**, in one file each, indifferent to
  rotation.

`daq.marker.n_marks()` reads **0** during a long run and that is not a fault:
it reports the last one-shot `acquire_with_pps()` record, while `long_run`
writes markers straight to disk. The file is the live count.

Interrupt this cell to stop watching — **the run continues**.

In [ ]:
def counts(outdir):
    """What is on disk right now. i64 files are 8 bytes per value."""
    def n_i64(name):
        path = outdir / name
        return path.stat().st_size // 8 if path.exists() else 0
    return (len(list(outdir.glob("*.f32"))), len(list(outdir.glob("*.h5"))),
            n_i64("edges.i64"), n_i64("markers.i64"))

try:
    while run.alive:
        s = run.status()
        f32, h5, n_edges, n_marks = counts(OUTDIR)
        print(f"{s['seconds']:8,.0f} s  seg {s['segment']:>3}  "
              f"{f32:>3} .f32  {h5:>3} .h5  "
              f"{n_edges:>7,} pps  {n_marks:>9,} marks")
        time.sleep(15)
except KeyboardInterrupt:
    print("stopped watching - the run continues")

f32, h5, n_edges, n_marks = counts(OUTDIR)
print(f"\nalive={run.alive}: {f32} .f32, {h5} .h5, {n_edges:,} pps, "
      f"{n_marks:,} marks")

### Stop

`run.join()` waits for `hours` to be reached; `run.stop()` ends it early. Either
way the **current segment is closed cleanly** and recorded in the manifest, and
the last segment is queued for conversion — so a run you cut short is complete,
just shorter.

In [ ]:
run.join()             # or run.stop() to end it early
daq.ao.stop()          # waveform off, ao0 back to 0 V
time.sleep(3)          # let the HDF5 worker drain the last segment
print("alive:", run.alive)
print("on disk:", counts(OUTDIR), "(.f32, .h5, pps, marks)")

### Read it back — four checks

Nothing below touches hardware, so it runs on any machine with the files.

1. **rotation lost nothing** — the segments of each channel are contiguous in
   scan index and sum to the same total
2. **every marker sits in exactly one segment**, found by manifest lookup
3. **the phase** — where the generator's edges fall inside the atomic second,
   and how far that walked over the run. This is the measurement the markers
   exist for
4. **every `.h5` is its `.f32`**, byte for byte, with the metadata attached

In [ ]:
manifest, edges = load_long_run(OUTDIR)
seg_rows = [e for e in manifest if "file" in e]
gaps = [e for e in manifest if e.get("event") == "gap"]
rate = manifest[0]["nominal_rate"]
channels = channels_of(OUTDIR)
n_total = sum(r["n"] for r in seg_rows if r["channel"] == channels[0])

print(f"{len(seg_rows)} segment files over {len(channels)} channel(s) "
      f"{channels}")
print(f"{n_total:,} scans = {n_total / rate / 60:.3f} min per channel")
print(f"{len(gaps)} gap(s)"
      + ("" if not gaps else "  <- the run RESTARTED; the samples either side "
                             "are NOT contiguous in time"))

# ---- 1. rotation lost nothing -----------------------------------------
for ch in channels:
    rows = sorted((r for r in seg_rows if r["channel"] == ch),
                  key=lambda r: r["index"])
    assert [r["index"] for r in rows] == list(range(len(rows))), \
        f"{ch}: segment indices are not 0..N-1, they are " \
        f"{[r['index'] for r in rows]}"
    assert sum(r["n"] for r in rows) == n_total, f"{ch}: scans missing"
    for a, b in zip(rows, rows[1:]):
        assert a["start_sample"] + a["n"] == b["start_sample"], \
            f"{ch}: segment {b['index']} does not start where {a['index']} ended"
    print(f"1. {ch}: {len(rows)} segments, contiguous, "
          f"{[r['n'] for r in rows]} scans")

# ---- 2. every marker sits in exactly one segment ----------------------
# Count MARKERS, not distinct scan indices. Repeated values are ordinary here -
# the pre-arm zeros are all scan 0 - so anything keyed on the scan index
# collapses them and then mis-reports the duplicates as unaccounted for.
marks, t_marks = marker_times(OUTDIR)
rows0 = sorted((r for r in seg_rows if r["channel"] == channels[0]),
               key=lambda r: r["start_sample"])

seg_of_mark = np.full(len(marks), -1, dtype=np.int64)      # -1 = no segment
for r in rows0:
    in_this = (marks >= r["start_sample"]) & (marks < r["start_sample"] + r["n"])
    assert not (seg_of_mark[in_this] >= 0).any(), \
        "a marker falls in two segments - the segments overlap"
    seg_of_mark[in_this] = r["index"]

inside = int((seg_of_mark >= 0).sum())
per_seg = {r["index"]: int((seg_of_mark == r["index"]).sum()) for r in rows0}
print(f"2. {len(marks):,} markers, {inside:,} inside a segment; "
      f"per segment {per_seg}")
if inside < len(marks):
    print(f"   ({len(marks) - inside:,} outside every segment - latched after "
          f"the last scan was written, or before scan 0)")

In [ ]:
# ---- 3. THE PHASE between the generator and the rubidium ---------------
# What the marker counter is for here. origin="edge" measures from the first
# ATOMIC SECOND instead of from the trigger, so the fractional part of a marker
# time is literally "where inside its second did that generator edge fall".
marks_all = load_markers(OUTDIR)
# Drop the edges that arrived while the AI task was arming - they all latched
# scan 0 and are not timestamps. See "The leading zeros" in 5.1.
n_zeros = int((marks_all == 0).sum())
marks_raw = marks_all[n_zeros:]
if n_zeros > 1:
    print(f"   ({n_zeros:,} pre-arm edges at scan 0 dropped)")

if len(marks_raw) < 3:
    print(f"3. only {len(marks_raw)} usable marker(s) - nothing to measure. Is "
          f"the generator running, and reaching the 2.5 V PFI threshold?")
else:
    t_edge = times_from_edges(marks_raw, edges, scans_per_second=rate,
                              origin="edge")
    frac = np.mod(t_edge, 1.0)          # position within the atomic second
    span = t_edge[-1] - t_edge[0]

    # HOW MANY PERIODS ELAPSED. Counted from the gaps, NOT from
    # round(span * MARKER_HZ): that assumes the generator is on frequency, and
    # breaks the moment the accumulated drift passes half a period - 0.5 ms at
    # 1 kHz, which 20 ppm reaches in four minutes. Rounding each gap to a whole
    # number of periods is the same trick _atomic_seconds uses on the 1 pps, and
    # it survives both drift and a missed edge.
    gaps = np.diff(marks_raw)
    period_scans = float(np.median(gaps))
    steps = np.rint(gaps / period_scans)
    n_periods = int(steps.sum())
    missed = int((steps > 1).sum())
    repeats = int((gaps == 0).sum())

    # WHICH period each edge is, counted CUMULATIVELY. The plot below draws the
    # phase against this, and it must not be round(elapsed * MARKER_HZ): that
    # assigns each edge to its NEAREST nominal period, so the residual is phase
    # modulo ONE PERIOD and wraps every time the drift passes half of one. At
    # 1 kHz that is 500 us, which a few ppm crosses in a minute - the plot came
    # out a sawtooth whose vertical drops meant nothing. Cumulative cannot wrap.
    period_index = np.concatenate([[0.0], np.cumsum(steps)])
    # how far AHEAD of a perfect MARKER_HZ each edge arrived, in seconds
    lead = period_index / MARKER_HZ - (t_edge - t_edge[0])

    f_meas = n_periods / span
    ppm = (f_meas - MARKER_HZ) / MARKER_HZ * 1e6
    # seconds the generator GAINED on the rubidium: positive = it ran fast, so
    # its edges arrived earlier than a perfect MARKER_HZ would have put them.
    gain = n_periods / MARKER_HZ - span

    print(f"3. {len(marks_raw):,} usable edges over {span:,.3f} atomic s")
    print(f"   the TRIGGER sat {frac[0]:.6f} s into its atomic second "
          f"- the phase you cannot know in advance")
    print(f"   median period {period_scans:,.1f} scans "
          f"({period_scans / rate * 1e6:,.1f} us), {n_periods:,} periods "
          f"elapsed{f', {missed:,} with a MISSED edge' if missed else ''}")
    print(f"   {f_meas:,.6f} Hz measured vs {MARKER_HZ:g} Hz set "
          f"({ppm:+,.2f} ppm)")
    print(f"   the generator GAINED {gain * 1e6:+,.1f} us on the rubidium "
          f"({gain / span * 1e6:+,.3f} us per second)")
    print(f"   resolution: one scan = {1e6 / rate:.1f} us, so ONE edge is "
          f"placed to about +/-{1e6 / rate / 2:.1f} us;")
    print(f"   averaging {len(marks_raw):,} of them over {span / 60:.1f} min "
          f"is what buys the ppm")

    # IS THE OFFSET CONSTANT? The ppm above is one number for the whole run -
    # the average. If the generator wanders, the average can sit near zero while
    # the phase swings widely, and then the ppm and the plot below tell
    # different stories. The scatter about a straight line is what distinguishes
    # them, so it is printed rather than left to be noticed on the plot.
    fit = np.polyfit(t_edge, lead, 1)
    resid = lead - np.polyval(fit, t_edge)
    print(f"   drift fit: {fit[0] * 1e6:+,.2f} ppm slope, scatter about the "
          f"line {resid.std() * 1e6:,.1f} us "
          f"(one scan is {1e6 / rate:.0f} us)")
    if resid.std() > 2.0 / rate:
        print(f"   !! that scatter is much bigger than the resolution, so the "
              f"offset is NOT steady - the generator is wandering, and the "
              f"single ppm above is only its average. Read the slope and the "
              f"plot, not the average")

    # what the gaps actually look like, so anomalies are visible not silent
    vals, cnt = np.unique(gaps, return_counts=True)
    order = np.argsort(cnt)[::-1][:6]
    print(f"   gap histogram (scans: count) "
          + ", ".join(f"{int(vals[i])}: {int(cnt[i]):,}" for i in order))

    if abs(ppm) > 50_000:
        print(f"   !! MARKER_HZ does not match the generator. Set "
              f"MARKER_HZ = {f_meas:,.0f} and re-run - the ppm is meaningless "
              f"until you do")
    # THE RESOLUTION TEST. Two markers at the same scan index mean two edges
    # inside one scan period, and the period count above cannot be trusted.
    if repeats and period_scans < 4:
        print(f"   !! {repeats:,} pairs share a scan index and the median gap "
              f"is only {period_scans:,.1f} scans - the generator is at or "
              f"past one edge per scan ({rate:,.0f}/s is the ceiling). No "
              f"single edge is resolved; slow it down")
    elif repeats:
        print(f"   {repeats:,} pairs share a scan index while the median gap "
              f"is {period_scans:,.0f} scans, so these are DUPLICATE counter "
              f"reads (or the clock stopped), not two edges in one scan. A "
              f"zero gap contributes no period, so the ppm above still stands")
    else:
        print(f"   no two markers share a scan index, so every edge is "
              f"individually resolved")

In [ ]:
# ---- 4. every .h5 is its .f32, with the metadata attached -------------
try:
    import h5py
except ImportError:
    print("4. h5py not installed - raw .f32 only, nothing to check")
else:
    late = [r for r in seg_rows
            if not (OUTDIR / r["file"]).with_suffix(".h5").exists()]
    if late:
        print(f"4. {len(late)} segment(s) not converted - finishing them now "
              f"with to_hdf5()")
        to_hdf5(OUTDIR)          # the after-the-fact path: same function

    bad, raw_bytes, h5_bytes = [], 0, 0
    for r in seg_rows:
        f32_path = OUTDIR / r["file"]
        h5_path = f32_path.with_suffix(".h5")
        raw = np.fromfile(f32_path, dtype=np.float32)
        with h5py.File(h5_path) as f:
            d = f["signal"]
            ok = (len(d) == r["n"] == len(raw)
                  and int(d.attrs["start_sample"]) == r["start_sample"]
                  and str(d.attrs["channel"]) == r["channel"]
                  and np.array_equal(d[:], raw))
        if not ok:
            bad.append(h5_path.name)
        raw_bytes += f32_path.stat().st_size
        h5_bytes += h5_path.stat().st_size

    print(f"4. {len(seg_rows) - len(bad)} of {len(seg_rows)} .h5 match their "
          f".f32 byte for byte, with start_sample/channel/n attached")
    if bad:
        print("   MISMATCHED:", bad)
    print(f"   {raw_bytes / 1e6:,.1f} MB raw -> {h5_bytes / 1e6:,.1f} MB gzip "
          f"({h5_bytes / raw_bytes * 100:.0f}% of the raw)")
    print("   the .f32 is KEPT: delete_raw defaults to False, and the manifest "
          "still names the .f32")

### The whole run on one axis

Segment boundaries, markers and the signal together — the picture that shows
rotation changed nothing. Read one segment at a time so a long run never has to
fit in RAM.

In [ ]:
fig, (ax, ax_p, ax2) = plt.subplots(3, 1, figsize=(13, 10))

# --- 1. THE WHOLE RUN, as a min/max envelope, one colour per segment.
# Plotting every sample of a 20-minute run is 30 million points that render as
# a solid block. The envelope keeps the EXTREMES of each block, which is what
# "did the signal ever drop out?" actually asks - and it is what the driver's
# own live buffer does, for the same reason.
for i, (idx_seg, y_seg) in enumerate(iter_segments(OUTDIR, channels[0],
                                                   mmap=True)):
    block = max(1, len(idx_seg) // 1000)         # ~1000 blocks per segment
    keep = (len(idx_seg) // block) * block
    blocks = np.asarray(y_seg[:keep]).reshape(-1, block)
    t = times_from_edges(idx_seg[:keep:block], edges, scans_per_second=rate)
    ax.fill_between(t, blocks.min(axis=1), blocks.max(axis=1), lw=0,
                    label=f"seg {i}")
    t0 = times_from_edges(np.array([idx_seg[0]]), edges,
                          scans_per_second=rate)[0]
    ax.axvline(t0, color="k", ls=":", lw=1)      # where rotation happened

# markers as a RUG along the top, so they annotate without hiding the signal
every = max(1, len(t_marks) // 200)
ax.plot(t_marks[::every], np.full(len(t_marks[::every]), 1.0), "|",
        color="red", ms=8, clip_on=False, transform=ax.get_xaxis_transform(),
        label=f"markers (1 in {every})")

ax.set_ylabel(f"{channels[0]} [V]  (min-max per block)")
ax.set_xlabel("atomic seconds since the trigger")   # NOTE: seconds, not minutes
ax.set_title(f"1. the whole run: {n_total / rate / 60:.1f} min, "
             f"{len(rows0)} segments, {len(marks):,} markers "
             f"- dotted lines are rotations")
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=8, ncol=len(rows0) + 1)

# --- 2. THE PHASE DRIFT: how far AHEAD of a perfect MARKER_HZ each generator
# edge arrived. `lead` and the fit both come from check 3, so the number in this
# title and the line under it cannot disagree.
#
# Why not plot the phase inside the atomic second directly? Because a few ppm is
# invisible against a 1000 ms axis. And why cumulative period indexing? Because
# the nearest-period version wraps at +/-half a period - 500 us at 1 kHz - and
# turns a straight drift into a meaningless sawtooth.
if len(marks_raw) >= 2:
    ax_p.plot(t_edge / 60, lead * 1e6, ".", ms=2, alpha=0.5,
              label="each generator edge")
    ax_p.plot(t_edge / 60, np.polyval(fit, t_edge) * 1e6, "r-", lw=1,
              label=f"{fit[0] * 1e6:+,.2f} ppm straight-line fit")
    ax_p.axhline(0.0, color="k", lw=0.8, ls=":")
    ax_p.set_ylabel("generator ahead of a\nperfect MARKER_HZ [us]")
    ax_p.set_xlabel("atomic minutes since the first 1 pps edge")
    ax_p.set_title(f"2. the phase: trigger at {frac[0]:.4f} s into its atomic "
                   f"second, then {gain * 1e6:+,.0f} us gained "
                   f"({ppm:+,.2f} ppm avg); {1e6 / rate:.0f} us steps = "
                   f"one scan", fontsize=10)
    ax_p.legend(fontsize=8)
    ax_p.grid(alpha=0.3)
else:
    ax_p.set_title("not enough markers to plot a phase")

# --- 3. ONE ROTATION, close up: the data is unbroken across the join.
# Read only the two segments either side, so this scales to an 8 h run.
if len(rows0) > 1:
    join = rows0[1]["start_sample"]
    # Wide enough to contain a full cycle of whatever ai0 is carrying, or the
    # window lands inside one half-period of the AO square and the panel is a
    # flat line - which proves continuity but shows nothing.
    zoom = int(max(0.01, 2.5 / daq.ao.freq()) * rate)
    a_idx, a_y = load_segment(OUTDIR, 0, channels[0], mmap=True)
    b_idx, b_y = load_segment(OUTDIR, 1, channels[0], mmap=True)
    idx_z = np.concatenate([a_idx[-zoom:], b_idx[:zoom]])
    y_z = np.concatenate([np.asarray(a_y[-zoom:]), np.asarray(b_y[:zoom])])

    t_z = times_from_edges(idx_z, edges, scans_per_second=rate)
    t_join = times_from_edges(np.array([join]), edges,
                              scans_per_second=rate)[0]
    ax2.plot((t_z - t_join) * 1e3, y_z, lw=1, marker=".", ms=3)
    ax2.axvline(0.0, color="k", ls=":", lw=1.5,
                label=f"seg 0 ends | seg 1 begins, at scan {join:,}")
    ax2.set_title("3. the same signal across a rotation - one file ends, the "
                  "next begins, and the samples keep their even spacing")
    ax2.set_xlabel("ms either side of the rotation")
    ax2.legend(fontsize=8)
else:
    ax2.set_title("only one segment - raise MINUTES or lower ROTATE_MINUTES "
                  "to see a rotation")

ax2.set_ylabel(f"{channels[0]} [V]")
ax2.grid(alpha=0.3)
fig.tight_layout()

### What each outcome tells you

| result | meaning |
|---|---|
| all four checks pass | rotation, markers and archiving work together — this is the configuration to leave overnight |
| **check 1 fails on segment indices** | the writer is not incrementing `index`; every manifest row would claim segment 0 and `load_segment` could only find the first |
| **check 1 fails on contiguity** | scans were dropped between segments — look for a `gap` row in the manifest first |
| `gaps` > 0 | the acquisition restarted. Both sides are real data, but they are **not** contiguous in time, and only the manifest records that. With `ai.trigger` set, a restarted task also waits for a trigger that may never come again — watch for `seconds` stuck at its old value |
| **repeated marker values** | either two pulses inside one scan period — the generator is too fast for 40 µs of resolution — or pulses that arrived while the AI clock was stopped, during a gap or before the trigger. Kept, not filtered: both are information |
| check 3: a few ppm, phase walking smoothly | the generator and the rubidium disagree by that much — which is the measurement. Nothing is wrong |
| check 3: the drift plot is a **40 µs staircase** | expected, and it is the resolution, not a fault: one edge is placed to ±half a scan. The slope through the staircase is still good to a fraction of it |
| check 3: **markers sharing a scan index** | two edges inside one scan period — the generator is faster than 1/scan can resolve, or the AI clock stopped. Their phase is not recoverable |
| check 3: **ppm in the thousands** | `MARKER_HZ` is not what the generator is actually set to. The period *count* no longer depends on it, so only the ppm comparison is wrong — set it to the measured Hz and re-run |
| check 3: **`n` periods with a MISSED edge** | the generator dropped a pulse, or it fell below the 2.5 V threshold. Counted as a gap rather than closed up, so the ppm stays honest |
| **`.h5` count stuck one behind** | normal *while running* — only closed segments convert. Stuck after the run means the worker died: re-run `to_hdf5(OUTDIR)` |
| **no `.h5` at all** | `h5py` missing. The worker logs the `ImportError` once and returns, and the run continues on raw `.f32` — by design, because losing the archive copy must not stop a week-long run |
| `.h5` barely smaller than `.f32` | gzip found no structure — normal for a noise-dominated channel. Compression is a bonus; the metadata is the reason to convert |

### Scaling this up

Everything above is the overnight configuration already; only the numbers
change:

```python
run = daq.long_run(OUTDIR, hours=8, rotate_minutes=60,
                   to_hdf5=True, background=True)
```

Eight hours at 25 kS/s × 2 channels is about 5.8 GB of raw, in 8 hourly
segments per channel that are converted and archivable **as the run goes**.
Two things to remember at that length:

* the marker counter is 32-bit and rolls over every **47.7 h** at 25 kS/s.
  `collect()` unwraps it, so this is handled — but only if you never restart the
  counter mid-run
* `marker_times` is exact for markers *near the first 1 pps edge*; a marker
  eight hours in is placed to about **27 ms** because choosing the zero slides
  the whole edge table. If you need better, subtract two calls instead of
  sliding — see the note in `times_from_edges`

---
# 6. Reading a run back

Everything here is in `postprocess`, which needs **no hardware and no
NI-DAQmx** — you can do this on any machine.

### `load_long_run(outdir)` → `(manifest, edges)`

* `manifest` — the parsed `manifest.jsonl`: a header row (`event="start"` with
  the channels and nominal rate), one row per **file**, and one row per **gap**
  where an error interrupted the run.
* `edges` — **every** 1 pps edge of the whole run, raw. One file for the run,
  not per segment and not per channel, because edges are global scan indices
  and the sample clock ticks once per scan whatever the channel count.

`gaps` must be 0. A gap means the acquisition restarted: the samples either
side are adjacent in scan index but **not** contiguous in time, and only the
manifest records that.

In [ ]:
manifest, edges = load_long_run(OUTDIR)
rate  = manifest[0]["nominal_rate"]
gaps  = [e for e in manifest if e.get("event") == "gap"]

print("channels:", channels_of(OUTDIR))
print("segments:", sorted({e["index"] for e in manifest if "file" in e}))
print("gaps    :", len(gaps), " <- must be 0")
for g in gaps:
    print(f"   at scan {g['sample']:,}: {g['error']}")

print()
report = edge_report(edges, nominal_rate=rate)

### Getting the samples

| function | returns |
|---|---|
| `load_segment(outdir, index, channel, mmap=False)` | **one** segment. `index` is the segment number — 0 is the first file, 1 the second |
| `iter_segments(outdir, channel)` | yields `(scans, samples)` per segment — walk a long run without holding it all |
| `load_run(outdir, channel)` | the whole channel concatenated. 360 MB per hour at 25 kS/s — check first |

All three return **global scan indices** alongside the samples, so the pieces
line up end to end. `mmap=True` leaves the samples on disk and pages in only
what you touch, so a multi-gigabyte segment plots without the RAM.

### `times_from_edges(scans, edges, scans_per_second=None, clean=True)`

Turns scan indices into seconds since the first atomic second. It interpolates
between the **surrounding edges** rather than dividing by a rate, which is what
absorbs the drift.

Two rules:

* **Pass the whole edge table**, never a slice. The numbering starts from the
  first edge it is given, so slicing to "the edges inside this segment" would
  throw away the seconds that came before and shift everything.
* **It cleans by default.** The raw table contains the counter's start value
  and the end-of-acquisition value; leaving them in shifted a 300 s record to
  1.000 … 301.427 s. `clean=False` opts out and raises if the table needs it.

In [ ]:
# mmap=True keeps a multi-gigabyte segment on disk instead of in RAM
idx, y = load_segment(OUTDIR, 0, "ai0", mmap=True)

t_atomic = times_from_edges(idx, edges, scans_per_second=rate)  # cleans by default
t_board  = idx / rate

print(f"{len(y):,} scans in segment 0")
print(f"atomic {t_atomic[0]:+.4f} .. {t_atomic[-1]:,.4f} s")
print(f"board  {t_board[0]:+.4f} .. {t_board[-1]:,.4f} s")

In [ ]:
# walk every segment without loading the whole run
for idx_s, y_s in iter_segments(OUTDIR, "ai0"):
    print(f"segment: scans {idx_s[0]:>12,} .. {idx_s[-1]:>12,}  "
          f"{len(y_s):>10,} samples  mean {y_s.mean(dtype=np.float64):+.4f} V")

In [ ]:
start, end = -10000, -1

plt.figure(figsize=(12, 4))
plt.plot(t_atomic[start:end], y[start:end], lw=0.8)
plt.xlabel("atomic time [s]"); plt.ylabel("V [V]"); plt.grid(alpha=0.3)

### The drift curve — what the run was really for

`np.diff(kept)` is scans per atomic second, second by second. That *is* the
board's clock being measured against an atomic reference, and it is the plot
worth keeping from a long run.

In [ ]:
kept, dropped = clean_edges(edges, rate)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(len(kept) - 1), np.diff(kept), lw=1)
ax.axhline(rate, color="k", ls="--", lw=1, label=f"nominal {rate:,.0f}")
ax.set_xlabel("atomic second"); ax.set_ylabel("scans / atomic second")
ax.legend(); ax.grid(alpha=0.3)

print(f"{len(dropped)} artefact(s) discarded: {dropped}")
print(f"clock error {report['ppm']:+.2f} ppm -> {report['seconds_per_day']:.2f} s/day")

---
# 7. Reference

### Configuring

| call | does |
|---|---|
| `daq.ai.setup(chans, rate=, duration=, trigger=, terminal_config=)` | configure the input, return the channels |
| `daq.ao.setup(channels_role=, shape=, freq=, amp=, offset=, rate=, duty=, trigger=)` | configure the output, return the channels |
| `daq.pps.terminal(...)`, `daq.pps.counter(...)` | the 1 pps line and which counter latches it |
| `daq.check(strict=False)` | validate everything without touching hardware |
| `daq.wiring()` | the screw-terminal table, printed and returned |
| `daq.reset()`, `daq.ao.reset()`, `daq.ai.reset()` | clear what accumulates: roles, levels, trigger, on_armed |

### Measuring

| call | does |
|---|---|
| `daq.ai.acquire()` | one record, `{channel: array}`. The only thing that measures |
| `daq.ai.acquire_chunks(duration=None)` | stream it instead, chunk by chunk, constant memory |
| `daq.acquire_with_pps()` | one record **plus** the atomic seconds inside it |
| `daq.clock_check(seconds=)` | measure the board's clock against the rubidium |
| `ai0.trace()`, `daq.ai.time_axis()`, `daq.ai.atom_time_axis()` | the QCoDeS parameter view of the last record |

### Generating

| call | does |
|---|---|
| `ao0.dc(v)` | a constant voltage, applied immediately |
| `daq.ao.start()` / `daq.ao.stop()` | start / stop the waveform |
| `with daq.ao.generating(**overrides):` | generate for a block, stop guaranteed |
| `daq.ao.onboard()` | True if the waveform loops from board memory |

### Recording and reading

| call | does |
|---|---|
| `daq.long_run(outdir, hours=, rotate_minutes=, background=)` | record to disk |
| `run.status()`, `run.alive`, `run.stop()` | the background handle |
| `daq.marker.terminal(...)` | record the scan index of every pulse on a line |
| `daq.marker.last_marks` | the marks of the last acquire_with_pps |
| `daq.live_preview()` | the last 60 s of one channel, from RAM |
| `LiveScope(outdir, channel, width)` | navigate the files while they grow |
| `load_long_run`, `load_segment`, `iter_segments`, `load_run` | read it back |
| `load_markers`, `marker_times`, `trigger_time` | the events |
| `edge_report`, `clean_edges` | check the 1 pps table before trusting it |
| `times_from_edges(idx, edges, rate, origin=)` | scan indices → atomic seconds. `origin="start"` (scan 0 — the trigger — and the default), `"edge"`, or any scan index |
| `daq.terminals()`, `daq.terminal_roles()`, `daq.pinout()` | which pin is what |
| `daq.describe()`, `daq.snapshot()` | what everything was set to |

# 8. When something goes wrong

| symptom | cause |
|---|---|
| an edit to the driver changes nothing | **restart the kernel** — Python caches imported modules |
| ppm is enormous, `t_atomic` starts at 1.0 | same: a stale kernel, so the edge cleaning is not running |
| `t_atomic - t_board` is a flat line hundreds of ms off zero | the two axes have different zeros — you built one with `origin="edge"`. See §3: that offset is the trigger-to-tick phase, not an error, and it buries a µs-scale drift curve |
| `need at least two 1 pps edges` | the record was too short to hold two ticks, so there is no atomic **scale** to build. Acquire for longer, or use `ai.time_axis()` and accept the board's clock |
| the live plot never updates | missing `clear_output`/`display`, or `plt.pause` on the inline backend |
| the live waveform looks ragged | you are looking at a decimated summary — use `LiveScope` or §5.3 |
| `gaps` is not 0 | usually `-200279` (the PC did not read fast enough) or `-200361` (USB could not drain the board) |
| AO underflow `-200621` | `daq.ao.onboard()` was False — lower `ao.rate` or raise `ao.freq` |
| few or no 1 pps edges | check `daq.wiring()` — is the pulse on the terminal you think? |
| `FileExistsError` on `long_run` | that directory already holds a run. Use a new one |
| `KeyError: no segment N` | the message lists the segments that do exist |
| overflow after adding a channel | the rate limit is **aggregate** — `daq.check()` does the arithmetic |
| no `markers.i64` in the run directory | `daq.marker.terminal` was never set — the file is only created on the first marker |
| a marker with no pulse under it | it arrived before the task armed. Send it from `daq.ai.set_on_armed(...)` |
| `marks[0]` is not 0 | the trigger and marker lines are different, so the trigger was not marked |

**Keep the raw output directory.** Every number in this notebook can be
recomputed from it.

In [ ]:
daq.close()